# Tome 12, Chapter 1: Number fields and their integers (Python companion)

Exercises first, solutions at the end. We let the computer find embeddings and signatures, compute rings of integers and indices with PARI/GP (through `cypari2`), test Stickelberger's theorem and Kronecker's theorem on thousands of examples, and finish with a computer-science question: how hard is it to compute $\mathcal O_K$? Requires `cypari2`, `sympy` and `numpy` (`pip install -r requirements.txt`).

In [ ]:
import random, time
import numpy as np
import cypari2
from sympy import Poly, symbols, ZZ
pari = cypari2.Pari()
pari.allocatemem(4 * 10**8, silent=True)
x = symbols('x')
def P(coeffs):
    # The PARI polynomial with the given coefficients, highest degree first.
    return pari('Pol(%s)' % list(coeffs))

## Exercises

**P1.1 (★) Signatures.** For $f=X^3-2$, $X^3-X-1$, $X^3-3X+1$, $X^4-10X^2+1$ and $X^5-X-1$, find the roots numerically, count the real ones ($r_1$) and the pairs of complex ones ($r_2$), and check that the sign of $\operatorname{disc}(f)$ is $(-1)^{r_2}$ (Proposition 1.16). Then check it on $2000$ random irreducible quintics. Why may we use $\operatorname{disc}(f)$ instead of $d_K$?

**P1.2 (★★) Indices.** For each $f$ below, compute $\operatorname{disc}(f)$, the field discriminant $d_K$ (`nfdisc`) and the index $[\mathcal O_K:\mathbb Z[\theta]]$ from the index formula (Proposition 1.17). Compare with Examples 1.20 and 1.21 and Exercises 1.8, 1.9 and 1.11: $X^3-2$, $X^3-10$, $X^3-X^2-2X-8$, $X^4-10X^2+1$. Print the integral basis PARI finds (`nfbasis`) and match it with the one in the book.

**P1.3 (★) Stickelberger in bulk.** Draw $3000$ random irreducible monic polynomials of degrees $2$ to $6$ with coefficients in $[-30,30]$. Check that $d_K\equiv0,1\pmod 4$ (Theorem 1.18) and that $\operatorname{disc}(f)\equiv0,1\pmod4$ too. Which residue is more frequent for $d_K$, and does it depend on the degree? Explain why the proof of Theorem 1.18 also covers $\operatorname{disc}(f)$.

**P1.4 (★★) Kronecker's theorem, found by brute force.** List all monic integer polynomials of degree at most $6$ with coefficients in $[-2,2]$, non-zero constant term, and all roots of absolute value at most $1$. Check that each one is a product of cyclotomic polynomials (Exercise 1.6). How close to $1$ can the largest root of an integer polynomial get otherwise? Find the polynomial in this box, not a product of cyclotomic polynomials, whose largest root has the smallest absolute value.

**P1.5 (★★★) CS application: how hard is $\mathcal O_K$?** *Dedekind's criterion* (Cohen, *A Course in Computational Algebraic Number Theory*, Theorem 6.1.4, stated here without proof; Chapter 2 proves the Kummer–Dedekind theorem, which rests on the same factorisation of $f$ modulo $p$) says: let $f\in\mathbb Z[X]$ be monic irreducible with root $\theta$ and $p$ a prime; factor $f\equiv\prod_i \bar g_i^{e_i}\pmod p$ with distinct monic irreducible $\bar g_i$; let $g=\prod g_i$ and $h=\prod g_i^{e_i-1}$ (any lifts to $\mathbb Z[X]$) and $F=(gh-f)/p$. Then $p\nmid[\mathcal O_K:\mathbb Z[\theta]]$ if and only if $\gcd(\bar F,\bar g,\bar h)=1$ in $\mathbb F_p[X]$.

1. Implement the criterion with `sympy` and check it against PARI's index on the four polynomials of P1.2 and on a few hundred random cubics, at every prime dividing $\operatorname{disc}(f)$.
2. The criterion is fast once you know the primes $p$ with $p^2\mid\operatorname{disc}(f)$. Take $f=X^2-p_1^2p_2$ with $p_1,p_2$ primes of $13$, $16$ and $19$ digits. Time `nfdisc(f)` and `nfdisc([f, 10^6])` (which only looks at primes up to $10^6$). Which one is right? What does this suggest about the cost of computing $\mathcal O_K$ in general?

## Solutions

### P1.1
**How to find it.** `numpy.roots` gives the roots; a root is real when its imaginary part is tiny. For the random test we need irreducible polynomials, which PARI checks with `polisirreducible`.

Since $\operatorname{disc}(f)=[\mathcal O_K:\mathbb Z[\theta]]^2\,d_K$, the two have the same sign, so the polynomial discriminant is enough.

In [ ]:
def signature(coeffs):
    r = np.roots(coeffs)
    r1 = int(sum(abs(z.imag) < 1e-9 for z in r))
    return r1, (len(r) - r1) // 2

for c in ([1, 0, 0, -2], [1, 0, -1, -1], [1, 0, -3, 1], [1, 0, -10, 0, 1], [1, 0, 0, 0, -1, -1]):
    r1, r2 = signature(c)
    D = int(pari.poldisc(P(c)))
    print(c, (r1, r2), D, (-1)**r2 == np.sign(D))

random.seed(1)
ok = n = 0
while n < 2000:
    c = [1] + [random.randint(-9, 9) for _ in range(5)]
    if not pari.polisirreducible(P(c)):
        continue
    n += 1
    r1, r2 = pari.polsturm(P(c)), None
    r2 = (5 - int(r1)) // 2
    ok += (-1)**r2 == int(pari.sign(pari.poldisc(P(c))))
print(ok, "of", n, "random quintics obey the sign rule")

For the random quintics we counted real roots exactly with Sturm sequences (`polsturm`), which avoids any rounding doubt. All $2000$ obey the rule. $X^3-3X+1$ is totally real with $\operatorname{disc}=81>0$; $X^4-10X^2+1$ too, with $147456=2304\cdot 8^2$.

### P1.2
**How to find it.** The index formula reads $\operatorname{disc}(f)=[\mathcal O_K:\mathbb Z[\theta]]^2 d_K$, so the index is $\sqrt{\operatorname{disc}(f)/d_K}$. PARI's `nfbasis` returns a basis in terms of $\theta$ (printed as `x`).

In [ ]:
for c in ([1, 0, 0, -2], [1, 0, 0, -10], [1, -1, -2, -8], [1, 0, -10, 0, 1]):
    f = P(c)
    D, dK = pari.poldisc(f), pari.nfdisc(f)
    print(f, "| disc(f) =", D, "| d_K =", dK, "| index =", pari.sqrtint(D // dK))
    print("    integral basis:", pari.nfbasis(f))

We find $d_K=-108$ (index $1$, Example 1.20), $-300$ (index $3$, basis $1,\theta,\frac{1+\theta+\theta^2}{3}$; the book's $\beta=\frac{\theta^2+\theta+1}3$, Example 1.21), $-503$ (index $2$, basis containing $\frac{\theta^2+\theta}2$ or an equivalent element, Exercise 1.11) and $2304$ (index $8$, Exercise 1.8). PARI may print a different but equivalent basis: two integral bases differ by a matrix in $\mathrm{GL}_n(\mathbb Z)$.

### P1.3
**How to find it.** Just loop. The proof of Theorem 1.18 only used that the $\omega_j$ are algebraic integers and that the two halves $P$, $N$ of the determinant are permuted by the Galois group; with $\omega_j=\theta^{j-1}$ that is $\operatorname{disc}(f)$.

In [ ]:
from collections import Counter
random.seed(2)
count = {n: Counter() for n in range(2, 7)}
bad = 0
done = 0
while done < 3000:
    n = random.randint(2, 6)
    c = [1] + [random.randint(-30, 30) for _ in range(n)]
    f = P(c)
    if not pari.polisirreducible(f):
        continue
    done += 1
    dK, D = int(pari.nfdisc(f)), int(pari.poldisc(f))
    bad += (dK % 4 not in (0, 1)) + (D % 4 not in (0, 1))
    count[n][dK % 4] += 1
print("violations:", bad)
for n in count:
    print(n, dict(count[n]))

No violations. In these samples $d_K\equiv1\pmod4$ is somewhat more frequent, in every degree, but the balance shifts with the degree: $d_K$ is even exactly when $2$ ramifies in $K$ (Chapter 2), and the chance of that is not the same for every degree. For quadratic fields from random polynomials, $d_K\equiv1$ when the squarefree part of $a^2-4b$ is $1\bmod4$.

### P1.4
**How to find it.** There are only $5^6$ polynomials of degree $6$ in the box. `polroots` gives the roots; PARI's `poliscycloprod` decides whether a polynomial is a product of cyclotomic polynomials.

In [ ]:
from itertools import product
found, best = 0, (10, None)
for n in range(1, 7):
    for tail in product(range(-2, 3), repeat=n):
        if tail[-1] == 0:
            continue
        f = P((1,) + tail)
        house = max(abs(complex(z)) for z in pari.polroots(f))
        if house <= 1 + 1e-9:
            found += 1
            assert pari.poliscycloprod(f), f
        elif house < best[0]:
            best = (house, f)
print(found, "polynomials with all roots in the closed unit disc; all are products of cyclotomic polynomials")
print("smallest largest root otherwise:", best)

All the polynomials found are products of cyclotomic polynomials, as Kronecker's theorem predicts. The smallest "house" (largest absolute value of a root) among the others in the box is just above $1$, and it comes from a polynomial with a real root $>1$ and the others inside the disc. How close to $1$ the house of a non-cyclotomic algebraic integer of degree $n$ can be is the *Schinzel–Zassenhaus problem*, solved by Dimitrov in 2019: the house is at least $2^{1/(4n)}$. Its cousin, *Lehmer's problem* about the Mahler measure, is still open; the record holder is $X^{10}+X^9-X^7-X^6-X^5-X^4-X^3+X+1$ with Mahler measure $1.17628\ldots$.

### P1.5
**How to find it.** Follow the recipe literally: `sympy` factors modulo $p$ (`Poly(..., modulus=p).factor_list()`), and the coefficients of $gh-f$ are divisible by $p$ because $gh\equiv f$. For part 2, think about what `nfdisc` must find: the largest square dividing $\operatorname{disc}(f)=4p_1^2p_2$.

In [ ]:
def dedekind_maximal(coeffs, p):
    # True when Z[theta] is p-maximal, i.e. p does not divide [O_K : Z[theta]].
    f = Poly(coeffs, x, domain=ZZ)
    g = h = Poly(1, x, domain=ZZ)
    for gi, e in Poly(coeffs, x, modulus=p).factor_list()[1]:
        gz = Poly(gi.as_expr(), x, domain=ZZ)
        g, h = g * gz, h * gz**(e - 1)
    F = Poly([c // p for c in (g * h - f).all_coeffs()], x, domain=ZZ)
    mod_p = lambda Q: Poly(Q.as_expr(), x, modulus=p)
    return mod_p(F).gcd(mod_p(g)).gcd(mod_p(h)).degree() == 0

def index(coeffs):
    f = P(coeffs)
    return int(pari.sqrtint(pari.poldisc(f) // pari.nfdisc(f)))

def primes_of_disc(coeffs):
    return [int(p) for p in pari.factor(pari.poldisc(P(coeffs)))[0] if p > 1]

for c in ([1, 0, 0, -2], [1, 0, 0, -10], [1, -1, -2, -8], [1, 0, -10, 0, 1]):
    print(c, "index", index(c), [(p, dedekind_maximal(c, p)) for p in primes_of_disc(c)])

random.seed(3)
checked = disagree = 0
for _ in range(300):
    c = [1] + [random.randint(-20, 20) for _ in range(3)]
    if not pari.polisirreducible(P(c)):
        continue
    ind = index(c)
    for p in primes_of_disc(c):
        checked += 1
        disagree += dedekind_maximal(c, p) != (ind % p != 0)
print(checked, "tests,", disagree, "disagreements")

In [ ]:
for k in (12, 15, 18):
    p1, p2 = pari.nextprime(7 * 10**k), pari.nextprime(3 * 10**k)
    f = pari('x^2 - %d' % (p1**2 * p2))
    t = time.perf_counter(); full = pari.nfdisc(f); t_full = time.perf_counter() - t
    t = time.perf_counter(); part = pari.nfdisc([f, 10**6]); t_part = time.perf_counter() - t
    print("%2d digits: full %.3fs, d_K right: %s | partial %.4fs, same answer: %s"
          % (k + 1, t_full, full in (p2, 4 * p2), t_part, part == full))

Dedekind's criterion agrees with PARI's index at every prime: at $3$ for $X^3-10$, at $2$ for Dedekind's cubic and for $X^4-10X^2+1$, and on all the random cubics.

Part 2 is the real lesson. Here $\mathbb Q(\sqrt{p_1^2p_2})=\mathbb Q(\sqrt{p_2})$ with $d_K=p_2$ or $4p_2$ according as $p_2\equiv1$ or $3\pmod 4$. The partial computation is instant, but it returns $p_1^2$ times the right answer: without factoring, nothing tells it that $p_1^2$ is a square factor, so it believes $\mathbb Z[\sqrt{p_1^2p_2}]$ is maximal. The full computation is right, and its time grows quickly with the size of $p_1$, because it has to factor $\operatorname{disc}(f)$.

In general, computing $\mathcal O_K$ is polynomial-time equivalent to finding the largest square dividing an integer (Chistov 1989; Buchmann and Lenstra, *Approximating rings of integers in number fields*, 1994). Nobody knows how to do that without factoring. The algorithms used in practice (the Round 2 and Round 4 algorithms) are fast at each prime $p$, as Dedekind's criterion is. The hard part is finding the primes $p$ with $p^2\mid\operatorname{disc}(f)$.